# NRPy 2 in Ten Minutes

NRPy 2 turns SymPy expressions, tensor component lists, gridfunction metadata, and finite-difference operators into generated C kernels. This overview shows the core workflow used throughout the later wave-equation and curvilinear chapters.

In [ ]:
import sympy as sp

import nrpy.c_codegen as ccg
import nrpy.finite_difference as fd
import nrpy.indexedexp as ixp

## Symbolic objects

Indexed expressions are nested Python lists of SymPy objects. They make tensor structure explicit while keeping components easy to pass to SymPy and NRPy code generation.

In [ ]:
phi_dDD = ixp.declarerank2("phi_dDD", symmetry="sym01")
laplacian_phi = sum(phi_dDD[i][i] for i in range(3))

print("Independent components in phi_dDD:")
print(phi_dDD)
print("Flat-space symbolic Laplacian:")
print(laplacian_phi)

## Generated code

`nrpy.c_codegen.c_codegen` converts symbolic expressions to C assignments. Here the wave speed is a symbolic parameter in the expression; later notebooks show how to register generated-code parameters explicitly.

In [ ]:
c = sp.Symbol("c", real=True)
rhs_phi = c**2 * laplacian_phi

print(ccg.c_codegen(rhs_phi, "rhs_phi", include_braces=False, verbose=False))

## Finite-difference stencils

Finite-difference operators are named by derivative type and coordinate direction. The main finite-difference notebook connects these coefficients to gridfunction reads and generated helper functions.

In [ ]:
for operator in ["dD0", "dDD00", "dupD0"]:
    coeffs, stencil = fd.compute_fdcoeffs_fdstencl(operator, fd_order=4)
    print(operator)
    for coeff, point in zip(coeffs, stencil):
        print(f"  {str(coeff):>5} at {point}")

## Where this leads

The Part 1 notebooks separate the pieces used in complete applications:

- `params`: Python-side parameters and generated-code parameters.
- `indexedexp`: symbolic tensors, symmetries, Levi-Civita objects, and matrix inversion.
- `grid`: gridfunction registration, metadata, parity, and memory access strings.
- `c_codegen` and `c_function`: generated assignments and reusable C functions.
- `finite_difference`: derivative notation, stencils, gridfunction reads, and FD helper functions.
- `reference_metric`: coordinate maps, scale factors, and metric data for curvilinear systems.